# Preprocess Rubric

Using Qwen 3.8 27B and Using `ECE_150_Square_Root_Marmoset_Tests.pdf` to generate a json output.


In [1]:
import json

from core.log import log_success, log_fail, log_warning, log_info

from core.io import read_prompt, pdf_to_image, image_to_messages, write_json

from preproc_rubric.preproc_rubric_llm import run_qwen3_8


In [2]:
prompt = read_prompt(prompt_path="../src/preproc_rubric/prompts/rubric_extraction.md")

imgs = pdf_to_image(pdf_path="../src/preproc_rubric/data/ECE_150_Square_Root_Marmoset_Tests.pdf")

msg = image_to_messages(images=imgs, prompt=prompt)

response = run_qwen3_8(messages=msg, temperature=0.0)

rubric_out = json.loads(response) if isinstance(response, str) else response

if not isinstance(rubric_out, dict):
    log_fail("The LLM response does not contain a JSON object.")
    raise ValueError("The LLM response must contain a JSON object.")

write_json(data=response, json_path="data/rubric_out.json")


[2026-09-17 19:52:16] [ INFO  ] Reading prompt: ../src/preproc_rubric/prompts/rubric_extraction.md
[2026-09-17 19:52:16] [SUCCESS] Prompt loaded successfully: 14075 characters
[2026-09-17 19:52:16] [ INFO  ] Converting PDF to images: ../src/preproc_rubric/data/ECE_150_Square_Root_Marmoset_Tests.pdf
[2026-09-17 19:52:16] [ INFO  ] DPI: 200
[2026-09-17 19:52:16] [ INFO  ] Rendered 4 page images
[2026-09-17 19:52:16] [SUCCESS] PDF processing complete: 4 page images
[2026-09-17 19:52:16] [ INFO  ] Building LLM messages from 4 page images
[2026-09-17 19:52:17] [SUCCESS] Built 2 messages containing 4 page images
[2026-09-17 19:52:17] [ INFO  ] Preparing request for model: preproc-rubric
[2026-09-17 19:52:17] [ INFO  ] Base URL: http://localhost:8000
[2026-09-17 19:52:17] [ INFO  ] Temperature: 0.0
[2026-09-17 19:52:17] [ INFO  ] Max tokens: default
[2026-09-17 19:52:17] [ INFO  ] Message count: 2
[2026-09-17 19:52:17] [ INFO  ] Sending request to LLM server...
[2026-09-17 21:08:11] [ INFO  ]

PosixPath('data/rubric_out.json')

After the payload is generated, we can process the json file into different sections.

In [3]:
from preproc_rubric.proc_rubric_json import load_rubric


In [4]:
rubric = load_rubric(json_path="data/rubric_out.json")

public_tests = rubric.get("public_tests")
release_tests = rubric.get("release_tests")
secret_tests = rubric.get("secret_tests")
scoring = rubric.get("scoring")
requirements = rubric.get("requirements")
issues = rubric.get("issues")
feedback_issue_types = rubric.get("feedback_issue_types")
execution_policy = rubric.get("execution_policy")

log_info(f"public_tests: {public_tests}")

log_info(f"release_tests: {release_tests}")

log_info(f"secret_tests: {secret_tests}")

log_info(f"scoring: {scoring}")

log_info(f"requirements: {requirements}")

log_warning(f"issues: {issues}")

log_info(f"feedback_issue_types: {feedback_issue_types}")

log_info(f"execution_policy: {execution_policy}")



[2026-09-17 21:08:11] [ INFO  ] Reading JSON: data/rubric_out.json
[2026-09-17 21:08:11] [WARNING] Double-encoded JSON detected; decoding once more.
[2026-09-17 21:08:11] [SUCCESS] JSON loaded successfully.
[2026-09-17 21:08:11] [WARNING] The rubric is marked as requiring review.
[2026-09-17 21:08:11] [SUCCESS] Processed 28 tests into 3 groups.
[2026-09-17 21:08:11] [ INFO  ] Building rubric lookup.
[2026-09-17 21:08:11] [SUCCESS] Rubric lookup created successfully.
[2026-09-17 21:08:11] [ INFO  ] public_tests: [{'id': 'P100', 'group_id': 'G001', 'description': "Check for an explicit 'return 0;' on normal completion of main (course convention).", 'graded': True, 'max_points': 1, 'visibility': 'student_visible', 'requirement_ids': ['REQ022'], 'pass_rule': "Source contains an explicit 'return 0;' on normal completion of main.", 'runs': [], 'generation_spec': None, 'sources': [{'page': 1, 'section': 'Public tests /7', 'evidence': 'P100_explicit_return: Check for an explicit return 0; on n